In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Library loans

A public library exported its loan records for the quarter. Clean the data and answer the questions below — no steps provided.

- One loan entry is a true duplicate.
- `genre` and `returned` have inconsistent casing.

After cleaning:
- Which genre has been checked out the most?
- Which member has borrowed the most books?
- What is the average days out per genre? Use `np.mean` inside a groupby.
- Use `np.unique` to list the distinct genres after standardizing.

In [10]:
loans = pd.DataFrame({
    'loan_id':   ['L001','L002','L003','L004','L005','L006','L007','L008','L002'],
    'member_id': ['M10','M20','M30','M10','M40','M20','M30','M10','M20'],
    'title':     ['Dune','1984','Sapiens','The Road','1984','Dune','The Road','Sapiens','1984'],
    'genre':     ['Sci-Fi','Dystopia','Non-Fiction','Literary Fiction','DYSTOPIA','sci-fi','literary fiction','non-fiction','Dystopia'],
    'days_out':  [14, 21, 30, 10, 21, 18, 12, 28, 21],
    'returned':  ['Yes','No','YES','yes','No','YES','no','Yes','No'],
})

# Your code here
d = loans.drop_duplicates().copy()
d['genre'] = d['genre'].str.lower()
d['returned'] = d['returned'].str.lower()

dc = d['genre'].value_counts()
print(dc.index[dc==dc.max()],'are cheked out the most')

dcm = d['member_id'].value_counts()
print(dcm.index[dcm==dcm.max()],'borrowed most')

print(d.groupby('genre')['days_out'].apply(lambda x: np.mean(x)))
print(np.unique(d['genre']))


Index(['sci-fi', 'dystopia', 'non-fiction', 'literary fiction'], dtype='object', name='genre') are cheked out the most
Index(['M10'], dtype='object', name='member_id') borrowed most
genre
dystopia            21.0
literary fiction    11.0
non-fiction         29.0
sci-fi              16.0
Name: days_out, dtype: float64
['dystopia' 'literary fiction' 'non-fiction' 'sci-fi']


---

## Level 2 — Gym memberships

A fitness studio exported its member list. Clean it, then answer the questions.

Known issues:
- One exact duplicate row
- `plan`: inconsistent casing
- `fee`: stored as `'$89.99'` strings
- `sessions`: strings; some are `'n/a'`
- `active`: mixed `'TRUE'`/`'False'`/`'true'` strings — convert to boolean

After cleaning:

1. What fraction of members are currently active?
2. Which plan type has more members — basic or premium?
3. Is there a correlation between sessions attended and monthly fee? Use `np.corrcoef` (drop rows with missing sessions first).
4. What is the 75th percentile number of sessions attended? Use `np.nanpercentile`.
5. Among active members only, who has attended the most sessions?

In [29]:
members = pd.DataFrame({
    'member_id': ['G001','G002','G003','G004','G005','G006','G007','G008','G009','G010','G011','G004'],
    'name':      ['Lena Park','James Wu','Tom Reed','Sara Bell','Mark Lane',
                  'Lena Park','James Wu','Tom Reed','Sara Bell','Mark Lane','Chris Day','Sara Bell'],
    'plan':      ['PREMIUM','basic','Premium','Basic','PREMIUM',
                  'premium','BASIC','premium','BASIC','basic','Premium','Basic'],
    'fee':       ['$89.99','$39.99','$89.99','$39.99','$89.99',
                  '$89.99','$39.99','$89.99','$39.99','$39.99','$89.99','$39.99'],
    'sessions':  [24, 8, 15, 30, 12, 22, 'n/a', 18, 'n/a', 14, 20, 30],
    'active':    ['TRUE','FALSE','True','TRUE','False','TRUE','False','True','TRUE','False','TRUE','TRUE'],
})

# Your code here


m = members.drop_duplicates().copy()
m['plan'] = m['plan'].str.lower()
m['fee'] = m['fee'].str.replace('$','',regex = False)
m['fee'] = m['fee'].str.replace(',','',regex = False)
m['fee'] = pd.to_numeric(m['fee'])
m['sessions'] = pd.to_numeric(m['sessions'], errors = 'coerce')
m['active'] = m['active'].str.lower()
m['active'] = m['active'].map({'true': True, 'false': False})
print(m['active'].mean(),'are currently active')
print(m.groupby('plan')['name'].count().idxmax(),'has more members')
md = m.dropna(subset = ['fee','sessions'])
cc = np.corrcoef(md['sessions'],md['fee'])[0,1]
print('correlation is',cc)
print('not correlated')
print(np.nanpercentile(m['sessions'],75))
ma = m.loc[m['active']]
print(ma.loc[ma['sessions'].idxmax(),'member_id'],'has attented the most sessions')

0.6363636363636364 are currently active
premium has more members
correlation is 0.08685104172245643
not correlated
22.0
G004 has attented the most sessions


---

## Level 3 — Freelance projects

Twelve project records across three freelancers and four clients. One row is a duplicate. Everything else needs cleaning — figure it out yourself.

Answer these five questions:

1. Which freelancer billed the most total hours?
2. What is the average hourly rate per client?
3. Add a `revenue` column (hours × rate). Which freelancer generated the most total revenue? Does it match the answer to question 1?
4. What are the 25th and 75th percentile revenue values per project? Use `np.nanpercentile`.
5. Rank freelancers by total revenue from lowest to highest using `np.argsort`. Who ranks first?

In [47]:
projects = pd.DataFrame({
    'project_id': ['P01','P02','P03','P04','P05','P06','P07','P08','P09','P10','P11','P03'],
    'client':     ['Acme Corp','Blue Sky','Acme Corp','TechNow','Blue Sky','ACME CORP',
                   'technow','Blue Sky','Acme Corp','TECHNOW','Blue Sky','Acme Corp'],
    'freelancer': ['Sam Wong','Sam Wong','PRIYA PATEL','priya patel','raj kumar',
                   'SAM WONG','Raj Kumar','PRIYA PATEL','raj kumar','Sam Wong','priya patel','PRIYA PATEL'],
    'hours':      [12, 8, 20, 15, 6, 18, 25, 'n/a', 10, 14, 'n/a', 20],
    'rate_usd':   ['$120','$95','$150','$110','$80','$120','$110','$150','$80','$120','$150','$150'],
    'status':     ['Done','Done','In Progress','Done','DONE','in progress',
                   'done','In Progress','Done','DONE','done','In Progress'],
    'month':      ['Jan','Jan','Feb','Jan','Mar','Feb','Mar','Feb','Apr','Apr','Apr','Feb'],
})

# Your code here
p = projects.drop_duplicates().copy()
p['client'] = p['client'].str.lower()
p['freelancer'] = p['freelancer'].str.lower()
p['hours'] = pd.to_numeric(p['hours'], errors='coerce')
p['rate_usd'] = p['rate_usd'].str.replace('$','',regex = False)
p['rate_usd'] = pd.to_numeric(p['rate_usd'], errors='coerce')
p['status'] = p['status'].str.lower()

#p['hours'] = p['hours'].fillna(0)
print(p.groupby('freelancer')['hours'].sum().idxmax(),'billed the most total hours')
print(p.groupby('client')['rate_usd'].mean())
p['revenue'] = p['hours']*p['rate_usd']
pp = p.dropna(subset=['revenue','hours'])
print(np.corrcoef(pp['revenue'],pp['hours'])[0,1])
print('hours and revenue are highly correlated')

print(np.nanpercentile(p['revenue'],[25,75]))

g = pp.groupby('freelancer')['revenue'].sum()
print(g.index[np.argsort(g)])
print(g.index[np.argsort(g)][0],'rank first')

sam wong billed the most total hours
client
acme corp    117.500000
blue sky     118.750000
technow      113.333333
Name: rate_usd, dtype: float64
0.9546159482221842
hours and revenue are highly correlated
[ 800. 2160.]
Index(['raj kumar', 'priya patel', 'sam wong'], dtype='object', name='freelancer')
raj kumar rank first
